Now that you have the data, you'll need to transform it into a more readable format. For simplicity, the final class is located in the src/WeatherAPI.py file

In [11]:
import sys
from pathlib import Path

# since this file is in a folder, we cannot import things from other folders. this adds the 'src' folder into the system path so that we can import from it. 
sys.path.insert(0, str(Path.cwd().parent / "src"))

from WeatherAPI import WeatherAPI

In [6]:
weather_api = WeatherAPI(latitude="37.7749", longitude="-122.4194")
current = weather_api.get_current_weather()

In [9]:
# lets transform so that we can add to a DataFrame or other data structure for further analysis

from pydantic import BaseModel
from typing import Optional
from datetime import datetime

class CurrentWeather(BaseModel):
    # look at the response and map the fields to the model
    # you will have to determine the appropriate types and optionality and can assign default values if desired. for this example, any field in the json response with a null value is mapped to an optional field in the model by using the `Optional` type hint.
    
    # in the json response, fields with date values are represented as strings. You can keep them as strings in the model or convert them to datetime objects 
    cloudCeiling: Optional[float]
    cloudCover: int
    cloudCoverPhrase: str
    dayOfWeek: str
    dayOrNight: str
    expirationTimeUtc: datetime
    iconCode: int
    iconCodeExtend: int
    obsQualifierCode: Optional[int]
    obsQualifierSeverity: Optional[int]
    precip1Hour: float 
    precip6Hour: float 
    precip24Hour: float 
    pressureAltimeter: float 
    pressureChange: float 
    pressureMeanSeaLevel: float 
    pressureTendencyCode: int
    pressureTendencyTrend: str
    relativeHumidity: int
    snow1Hour: float 
    snow6Hour: float 
    snow24Hour: float 
    sunriseTimeLocal: datetime 
    sunriseTimeUtc: datetime 
    sunsetTimeLocal: datetime 
    sunsetTimeUtc: datetime 
    temperature: int 
    temperatureChange24Hour: int 
    temperatureDewPoint: int 
    temperatureFeelsLike: int 
    temperatureHeatIndex: int 
    temperatureMax24Hour: int 
    temperatureMaxSince7Am: int 
    temperatureMin24Hour: int 
    temperatureWetBulbGlobe: int 
    temperatureWindChill: int 
    uvDescription: str 
    uvIndex: int 
    validTimeLocal: datetime 
    validTimeUtc: datetime 
    visibility: float 
    windDirection: int 
    windDirectionCardinal: str 
    windGust: Optional[float] 
    windSpeed: int 
    wxPhraseLong: str 
    wxPhraseMedium: str 
    wxPhraseShort: str 

current_weather_obj = CurrentWeather(**current)

# if you preview the response, youll notice that the date fields are now converted to datetime objects automatically without any additional manual conversion

CurrentWeather(cloudCeiling=None, cloudCover=46, cloudCoverPhrase='Partly Cloudy', dayOfWeek='Tuesday', dayOrNight='D', expirationTimeUtc=datetime.datetime(2026, 10, 27, 17, 14, 53, tzinfo=TzInfo(0)), iconCode=30, iconCodeExtend=3000, obsQualifierCode=None, obsQualifierSeverity=None, precip1Hour=0.0, precip6Hour=0.0, precip24Hour=0.0, pressureAltimeter=30.07, pressureChange=0.0, pressureMeanSeaLevel=1018.0, pressureTendencyCode=0, pressureTendencyTrend='Steady', relativeHumidity=64, snow1Hour=0.0, snow6Hour=0.0, snow24Hour=0.0, sunriseTimeLocal=datetime.datetime(2026, 10, 27, 7, 29, 51, tzinfo=TzInfo(-25200)), sunriseTimeUtc=datetime.datetime(2026, 10, 27, 14, 29, 51, tzinfo=TzInfo(0)), sunsetTimeLocal=datetime.datetime(2026, 10, 27, 18, 16, 38, tzinfo=TzInfo(-25200)), sunsetTimeUtc=datetime.datetime(2026, 10, 28, 1, 16, 38, tzinfo=TzInfo(0)), temperature=61, temperatureChange24Hour=0, temperatureDewPoint=49, temperatureFeelsLike=60, temperatureHeatIndex=61, temperatureMax24Hour=67, te

# Other models

The weather API returns several different payload shapes, but many fields are conceptually similar. We can use reusable base models to avoid duplicated validation/parsing logic.

Take a look at src/models.py

1) Single-record response model
CurrentWeather is used for the current conditions payload (one object, scalar fields). Pydantic automatically parses and validates types (including datetime fields).

2) Columnar response base model
Forecast payloads (15-minute, hourly, daily) are mostly columnar: each key contains a list.
ColumnarForecastModel provides shared behavior:
- equal-length list validation across all configured columns
- periods property (number of time steps/rows)
- to_rows() method to convert columnar arrays into row dictionaries

3) Specialized forecast models
Forecast15Minute and Forecast2DayHourly inherit from ColumnarForecastModel and define their own typed fields.
Daily forecasts use reusable models too:
- ForecastDaypart: nested day/night arrays
- DailyForecastBase: shared daily fields + daypart list
- Forecast3DayDaily / Forecast7DayDaily: concrete daily response models

4) Reusable transformation helper
to_rows(forecast) can be used across forecast classes to flatten validated columnar data.
You can also pass selected fields to return only the columns needed for analysis.

5) Common usage pattern
1. Pull JSON from WeatherAPI
2. Validate with the matching model using model_validate(...)
3. Use .periods and to_rows(...) for DataFrame-ready records

This gives us:
- strong typing
- automatic parsing and validation
- reusable transformations for downstream analytics

In [15]:
from models import *
import pandas as pd

# Convert the current weather data to a DataFrame
current = weather_api.get_current_weather()
current_weather = CurrentWeather.model_validate(current)
pd.DataFrame([current_weather.model_dump()]).head()

,cloudCeiling,cloudCover,cloudCoverPhrase,dayOfWeek,dayOrNight,expirationTimeUtc,iconCode,iconCodeExtend,obsQualifierCode,obsQualifierSeverity,...,validTimeLocal,validTimeUtc,visibility,windDirection,windDirectionCardinal,windGust,windSpeed,wxPhraseLong,wxPhraseMedium,wxPhraseShort
0,None,36,Partly Cloudy,Tuesday,D,2026-10-27 19:12:31+00:00,30,3000,None,None,...,2026-10-27 12:02:31-07:00,2026-10-27 19:02:31+00:00,8.0,270,W,None,5,Partly Cloudy,Partly Cloudy,P Cloudy


In [21]:
two_day = weather_api.get_2_day_hourly_forecast()
two_day_forecast = Forecast2DayHourly.model_validate(two_day)
pd.DataFrame(two_day_forecast.to_rows()).head()

,cloudCover,dayOfWeek,dayOrNight,expirationTimeUtc,iconCode,iconCodeExtend,precipChance,precipType,pressureMeanSeaLevel,qpf,...,validTimeLocal,validTimeUtc,visibility,windDirection,windDirectionCardinal,windGust,windSpeed,wxPhraseLong,wxPhraseShort,wxSeverity
0,40,Tuesday,D,1793128951,30,3000,0,rain,30.04,0.0,...,2026-10-27 13:00:00-07:00,1793131200,9.0,289,WNW,NaN,6,Partly Cloudy,P Cloudy,1
1,59,Tuesday,D,1793128951,30,3000,0,rain,30.02,0.0,...,2026-10-27 14:00:00-07:00,1793134800,10.0,278,W,NaN,7,Partly Cloudy,P Cloudy,1
2,53,Tuesday,D,1793128951,30,3000,0,rain,30.00,0.0,...,2026-10-27 15:00:00-07:00,1793138400,10.0,280,W,17.0,11,Partly Cloudy,P Cloudy,1
3,59,Tuesday,D,1793128951,30,3000,1,rain,30.00,0.0,...,2026-10-27 16:00:00-07:00,1793142000,10.0,284,WNW,21.0,13,Partly Cloudy,P Cloudy,1
4,48,Tuesday,D,1793128951,30,3000,1,rain,29.99,0.0,...,2026-10-27 17:00:00-07:00,1793145600,10.0,288,WNW,21.0,13,Partly Cloudy,P Cloudy,1


In [22]:
three_day = weather_api.get_3_day_daily_forecast()
three_day_forecast = Forecast3DayDaily.model_validate(three_day)
pd.DataFrame(three_day_forecast.to_rows()).head()

,calendarDayTemperatureMax,calendarDayTemperatureMin,dayOfWeek,expirationTimeUtc,moonPhase,moonPhaseCode,moonPhaseDay,moonriseTimeLocal,moonriseTimeUtc,moonsetTimeLocal,...,qpfRain,qpfSnow,sunriseTimeLocal,sunriseTimeUtc,sunsetTimeLocal,sunsetTimeUtc,temperatureMax,temperatureMin,validTimeLocal,validTimeUtc
0,66,53,Tuesday,1793128861,Waning Gibbous,WNG,16,2026-10-27 19:08:13-07:00,1793153293,2026-10-27 09:28:27-07:00,...,0.0,0.0,2026-10-27 07:29:41-07:00,1793111381,2026-10-27 18:16:29-07:00,1793150189,66,49,2026-10-27 07:00:00-07:00,1793109600
1,69,49,Wednesday,1793128861,Waning Gibbous,WNG,18,2026-10-28 20:02:09-07:00,1793242929,2026-10-28 10:43:58-07:00,...,0.0,0.0,2026-10-28 07:30:42-07:00,1793197842,2026-10-28 18:15:18-07:00,1793236518,69,51,2026-10-28 07:00:00-07:00,1793196000
2,67,51,Thursday,1793128861,Waning Gibbous,WNG,19,2026-10-29 21:06:00-07:00,1793333160,2026-10-29 11:52:37-07:00,...,0.0,0.0,2026-10-29 07:31:44-07:00,1793284304,2026-10-29 18:14:08-07:00,1793322848,67,52,2026-10-29 07:00:00-07:00,1793282400
3,66,52,Friday,1793128861,Waning Gibbous,WNG,20,2026-10-30 22:17:32-07:00,1793423852,2026-10-30 12:51:39-07:00,...,0.0,0.0,2026-10-30 07:32:46-07:00,1793370766,2026-10-30 18:13:00-07:00,1793409180,65,50,2026-10-30 07:00:00-07:00,1793368800


In [23]:
seven_day = weather_api.get_7_day_daily_forecast()
seven_day_forecast = Forecast7DayDaily.model_validate(seven_day)
pd.DataFrame(seven_day_forecast.to_rows()).head()

,calendarDayTemperatureMax,calendarDayTemperatureMin,dayOfWeek,expirationTimeUtc,moonPhase,moonPhaseCode,moonPhaseDay,moonriseTimeLocal,moonriseTimeUtc,moonsetTimeLocal,...,qpfRain,qpfSnow,sunriseTimeLocal,sunriseTimeUtc,sunsetTimeLocal,sunsetTimeUtc,temperatureMax,temperatureMin,validTimeLocal,validTimeUtc
0,67,52,Tuesday,1793130594,Waning Gibbous,WNG,16,2026-10-27 19:08:18-07:00,1.793153e+09,2026-10-27 09:28:37-07:00,...,0.0,0.0,2026-10-27 07:29:49-07:00,1793111389,2026-10-27 18:16:36-07:00,1793150196,67,49,2026-10-27 07:00:00-07:00,1793109600
1,68,49,Wednesday,1793130594,Waning Gibbous,WNG,18,2026-10-28 20:02:14-07:00,1.793243e+09,2026-10-28 10:44:07-07:00,...,0.0,0.0,2026-10-28 07:30:50-07:00,1793197850,2026-10-28 18:15:25-07:00,1793236525,68,51,2026-10-28 07:00:00-07:00,1793196000
2,68,51,Thursday,1793130594,Waning Gibbous,WNG,19,2026-10-29 21:06:05-07:00,1.793333e+09,2026-10-29 11:52:46-07:00,...,0.0,0.0,2026-10-29 07:31:52-07:00,1793284312,2026-10-29 18:14:15-07:00,1793322855,68,52,2026-10-29 07:00:00-07:00,1793282400
3,66,52,Friday,1793130594,Waning Gibbous,WNG,20,2026-10-30 22:17:38-07:00,1.793424e+09,2026-10-30 12:51:49-07:00,...,0.0,0.0,2026-10-30 07:32:54-07:00,1793370774,2026-10-30 18:13:06-07:00,1793409186,65,50,2026-10-30 07:00:00-07:00,1793368800
4,64,50,Saturday,1793130594,Waning Gibbous,WNG,21,2026-10-31 23:31:04-07:00,1.793515e+09,2026-10-31 13:39:16-07:00,...,0.0,0.0,2026-10-31 07:33:57-07:00,1793457237,2026-10-31 18:11:59-07:00,1793495519,64,50,2026-10-31 07:00:00-07:00,1793455200


In [24]:
fifteen_minute = weather_api.get_15_minute_forecast()
fifteen_minute_forecast = Forecast15Minute.model_validate(fifteen_minute)
pd.DataFrame(fifteen_minute_forecast.to_rows()).head()

,dayOfWeek,iconCode,iconCodeExtend,precipChance,precipRate,precipType,relativeHumidity,snowRate,temperature,temperatureFeelsLike,validTimeLocal,windDirection,windDirectionCardinal,windSpeed,wxPhraseLong,wxPhraseShort,wxSeverity
0,Tuesday,30,3000,0,0.0,rain,55,0.0,64,64,2026-10-27 12:45:00-07:00,287,WNW,6,Partly Cloudy,P Cloudy,1
1,Tuesday,30,3000,0,0.0,rain,55,0.0,64,64,2026-10-27 13:00:00-07:00,286,WNW,6,Partly Cloudy,P Cloudy,1
2,Tuesday,30,3000,0,0.0,rain,55,0.0,64,64,2026-10-27 13:15:00-07:00,283,WNW,7,Partly Cloudy,P Cloudy,1
3,Tuesday,30,3000,0,0.0,rain,55,0.0,64,64,2026-10-27 13:30:00-07:00,281,W,7,Partly Cloudy,P Cloudy,1
4,Tuesday,30,3000,0,0.0,rain,54,0.0,65,65,2026-10-27 13:45:00-07:00,278,W,7,Partly Cloudy,P Cloudy,1
